In [1]:
#A set of experiments designed to see how different gain metrics impacts the performance of CON-FOLD.
#We can use XG-Boost as a control
#There also exists data from FOLD-SE that I can compare to later (it uses a Magic Gini Impurity Metric)
#Key metrics:
#1.) Accuracy
#2.) Inverse Brier Score (a metric I designed earlier)
#3.) Time to run (either CPU time or Wall time), it would be great if we could measure/estimate FLOPs but this is not high priority.
#4.) Number of rules produced
#5.) Number of predictes (if we can easily get this)
#Let's try it with both fit and confidence_fit (we use the existing default pruning parameters for confidence fit)
# Let's use all of these datasets:

import numpy as np
import pandas as pd
import time
import os
from statistics import mean, stdev
from joblib import Parallel, delayed

# Import your project files
# Ensure they are in the python path
from algo import metric_list
from foldrm import Classifier
from utils import split_data, split_xy, get_scores, count_rules_in_model, num_predicates, prune_rules, get_inverse_brier_score
from datasets import wine, ecoli, weight_lifting, wall_robot, page_blocks, nursery, dry_bean

datasets = [wine, ecoli, weight_lifting, wall_robot, page_blocks, nursery, dry_bean]
dataset_names = ["Wine", "Ecoli", "Weight Lifting", "Wall Robot", "Page Blocks", "Nursery", "Dry Bean"]


# Display options for the final DataFrame
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

C:\Users\Lachlan McGinness\GithubRepositories\CONFOLD\algo.py:2: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.stats import binom


In [2]:
#Cell 2: Helper functions
def run_metric_trials(data, model_template, metric, num_trials=30):
    """
    Modified version of run_trials to accept a metric.
    This runs the standard FOLD-RM (fit method).
    """
    scores = []
    brier_scores = []
    rule_counts = []
    predicate_counts = []

    # Calculate num_classes once from the full dataset
    num_classes = len(pd.DataFrame(data).iloc[:, -1].unique())
    
    for _ in range(num_trials):
        model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
        data_train, data_test = split_data(data, ratio=0.8)
        X_test, Y_test = split_xy(data_test)
        
        # Train model with the specified metric
        model.fit(data_train, ratio=0.5, metric=metric, num_classes=num_classes)
        
        # Predict and evaluate
        Ystar_test_tuples = model.predict(X_test)
        Ystar_test = [y[0] for y in Ystar_test_tuples]
        score = get_scores(Ystar_test, data_test)
        brier_score = get_inverse_brier_score(Ystar_test_tuples, Y_test)
        scores.append(score)
        brier_scores.append(brier_score)
    
        rule_counts.append(count_rules_in_model(model))
        
        model.asp()
        predicate_counts.append(num_predicates(model))
    
    avg_score = mean(scores) if scores else 0
    std_score = stdev(scores) if len(scores) > 1 else 0
    avg_brier = mean(brier_scores) if brier_scores else 0 
    std_brier = stdev(brier_scores) if len(brier_scores) > 1 else 0
    avg_rules = mean(rule_counts) if rule_counts else 0
    std_rules = stdev(rule_counts) if len(rule_counts) > 1 else 0
    avg_preds = mean(predicate_counts) if predicate_counts else 0
    std_preds = stdev(predicate_counts) if len(predicate_counts) > 1 else 0
    
    return avg_score, std_score, avg_brier, std_brier, avg_rules, std_rules, avg_preds, std_preds

def run_confidence_metric_trials(data, model_template, metric, num_trials=30):
    """
    Modified version of run_improved_pruned_trials to accept a metric.
    This runs CON-FOLD (confidence_fit method).
    """
    scores = []
    brier_scores = []
    rule_counts = []
    predicate_counts = []

    # Calculate num_classes once from the full dataset
    num_classes = len(pd.DataFrame(data).iloc[:, -1].unique())
    
    for _ in range(num_trials):
        model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
        data_train, data_test = split_data(data, ratio=0.8)
        X_test, Y_test = split_xy(data_test)
        
        # Train model with the specified metric using confidence_fit
        model.confidence_fit(data_train, metric=metric, num_classes=num_classes)
        #model.rules = prune_rules(model.rules, confidence_threshold=0.75) # Default pruning
    
        # Predict and evaluate
        Ystar_test_tuples = model.predict(X_test)
        Ystar_test = [y[0] for y in Ystar_test_tuples]
        score = get_scores(Ystar_test, data_test)
        brier_score = get_inverse_brier_score(Ystar_test_tuples, Y_test)
        scores.append(score)
        brier_scores.append(brier_score)
    
        rule_counts.append(count_rules_in_model(model))
    
        model.asp()
        predicate_counts.append(num_predicates(model))
        
    avg_score = mean(scores) if scores else 0
    std_score = stdev(scores) if len(scores) > 1 else 0
    avg_brier = mean(brier_scores) if brier_scores else 0
    std_brier = stdev(brier_scores) if len(brier_scores) > 1 else 0
    avg_rules = mean(rule_counts) if rule_counts else 0
    std_rules = stdev(rule_counts) if len(rule_counts) > 1 else 0
    avg_preds = mean(predicate_counts) if predicate_counts else 0
    std_preds = stdev(predicate_counts) if len(predicate_counts) > 1 else 0
    
    return avg_score, std_score, avg_brier, std_brier, avg_rules, std_rules, avg_preds, std_preds

def run_single_experiment(dataset_info, metric, num_trials):
    """
    Runs both FOLD-RM and CON-FOLD for one dataset and one metric.
    Returns a list containing the two result dictionaries.
    """
    dataset_func, name = dataset_info
    print(f"--- Starting: Dataset='{name}', Metric='{metric}' ---")
    
    model_template, data = dataset_func()
    
    # --- Run standard 'fit' method (FOLD-RM) ---
    start_time = time.time()
    fr_avg_acc, fr_std_acc, fr_avg_brier, fr_std_brier, fr_avg_rules, fr_std_rules, fr_avg_preds, fr_std_preds = run_metric_trials(data, model_template, metric=metric, num_trials=num_trials)
    fr_time = (time.time() - start_time) / num_trials
    fold_rm_result = {
        "Dataset": name, "Metric": metric, "Fit Method": "FOLD-RM (fit)", "Avg Accuracy": fr_avg_acc, 
        "Std Accuracy": fr_std_acc, "Avg Brier Score": fr_avg_brier, "Std Brier Score": fr_std_brier, 
        "Avg Time": fr_time, "Num Rules Avg": fr_avg_rules, "Num Rules Std": fr_std_rules, 
        "Num Preds Avg": fr_avg_preds, "Num Preds Std": fr_std_preds
    }

    # --- Run 'confidence_fit' method (CON-FOLD) ---
    start_time = time.time()
    cf_avg_acc, cf_std_acc, cf_avg_brier, cf_std_brier, cf_avg_rules, cf_std_rules, cf_avg_preds, cf_std_preds = run_confidence_metric_trials(data, model_template, metric=metric, num_trials=num_trials)
    cf_time = (time.time() - start_time) / num_trials
    con_fold_result = {
        "Dataset": name, "Metric": metric, "Fit Method": "CON-FOLD (confidence_fit)", "Avg Accuracy": cf_avg_acc, 
        "Std Accuracy": cf_std_acc, "Avg Brier Score": cf_avg_brier, "Std Brier Score": cf_std_brier, 
        "Avg Time": cf_time, "Num Rules Avg": cf_avg_rules, "Num Rules Std": cf_std_rules, 
        "Num Preds Avg": cf_avg_preds, "Num Preds Std": cf_std_preds
    }
    
    print(f"--- Finished: Dataset='{name}', Metric='{metric}' ---")
    return [fold_rm_result, con_fold_result]

In [3]:



# --- Cell 3 - SETUP AND SMART RESUME ---
if __name__ == "__main__":
    datasets_with_names = list(zip(datasets, dataset_names))
    NUM_TRIALS = 300
    backup_file = 'experiment_results_backup.csv'

    results_list = []
    completed_tasks = set()

    if os.path.exists(backup_file):
        print(f"--- Found existing backup file: {backup_file} ---")
        try:
            results_df = pd.read_csv(backup_file)
            results_list = results_df.to_dict('records')
            if not results_df.empty:
                for index, row in results_df.iterrows():
                    completed_tasks.add( (row['Dataset'], row['Metric']) )
            print(f"--- Resuming. {len(completed_tasks)} Dataset/Metric combinations already completed. ---")
        except Exception as e:
            print(f"--- Could not read backup file, starting fresh. Error: {e} ---")
            results_list = []
            completed_tasks = set()

    # --- 3. CREATE THE TASK LIST (skipping completed tasks) ---
    tasks_to_run = []
    for dataset_info in datasets_with_names:
        name = dataset_info[1]
        for metric in metric_list:
            if (name, metric) not in completed_tasks:
                tasks_to_run.append({'dataset_info': dataset_info, 'metric': metric, 'num_trials': NUM_TRIALS})
    
    print(f"\n>>> Total new tasks to run: {len(tasks_to_run)} <<<\n")

    # --- 4. RUN TASKS IN PARALLEL ---
    # n_jobs=-1 means use all available CPU cores.
    # You can set it to a specific number, e.g., n_jobs=4 to use 4 cores.
    # 'verbose=10' will print progress updates.
    if tasks_to_run:
        # The Parallel call executes the function for each task in the list
        # It returns a list of results, e.g., [[res1, res2], [res3, res4], ...]
        nested_results = Parallel(n_jobs=-1, verbose=10)(
            delayed(run_single_experiment)(**task) for task in tasks_to_run
        )
        
        # Flatten the list of lists into a single list of results
        new_results = [item for sublist in nested_results for item in sublist]
        
        # Add the new results to our main list (which may contain resumed data)
        results_list.extend(new_results)

    # --- 5. SAVE FINAL RESULTS ---
    if tasks_to_run:
        print("\n--- All parallel tasks complete! Saving final results... ---")
        final_df = pd.DataFrame(results_list)
        final_df.to_csv(backup_file, index=False)
        print(f"--- All results have been successfully saved to {backup_file} ---")
    else:
        print("\n--- No new tasks to run. Everything is already complete. ---")




>>> Total new tasks to run: 77 <<<



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:   11.0s
[Parallel(n_jobs=-1)]: Done  22 out of  77 | elapsed:   42.0s remaining:  1.8min
[Parallel(n_jobs=-1)]: Done  30 out of  77 | elapsed: 22.7min remaining: 35.6min
[Parallel(n_jobs=-1)]: Done  38 out of  77 | elapsed: 24.4min remaining: 25.1min
[Parallel(n_jobs=-1)]: Done  46 out of  77 | elapsed: 35.9min remaining: 24.2min
[Parallel(n_jobs=-1)]: Done  54 out of  77 | elapsed: 36.4min remaining: 15.5min
[Parallel(n_jobs=-1)]: Done  62 out of  77 | elapsed: 43.4min remaining: 10.5min
[Parallel(n_jobs=-1)]: Done  70 out of  77 | elapsed: 121.2min remaining: 12.1min



--- All parallel tasks complete! Saving final results... ---
--- All results have been successfully saved to experiment_results_backup.csv ---


[Parallel(n_jobs=-1)]: Done  77 out of  77 | elapsed: 127.3min finished


In [4]:
# Cell 4: Process, Display, and Save Results (UPDATED FOR SEPARATE TABLES)

def process_and_display_results(df, fit_method_name, filename):
    """
    Helper function to pivot, average, format, display, and save results.
    """
    # --- 1. Pivot the DataFrame (Datasets as Columns) ---
    pivoted_df = df.set_index(['Metric', 'Fit Method', 'Dataset'])
    pivoted_df = pivoted_df.unstack(level='Dataset')
    pivoted_df = pivoted_df.swaplevel(0, 1, axis=1)
    pivoted_df.sort_index(axis=1, level=0, inplace=True)
    
    # --- 2. Calculate the Average Across All Datasets ---
    average_df = df.groupby(['Metric', 'Fit Method']).mean(numeric_only=True)
    average_df.columns = pd.MultiIndex.from_product([['Average'], average_df.columns])
    
    # --- 3. Combine Average and Pivoted DataFrames ---
    final_summary_df = pd.concat([average_df, pivoted_df], axis=1)
    
    # --- 4. Format the DataFrame for Display ---
    format_dict = {
        "Accuracy Avg": "{:.4f}",
        "Accuracy Std": "{:.4f}",
        "Brier Score Avg": "{:.4f}", # Added Brier format
        "Brier Score Std": "{:.4f}", # Added Brier format
        "Avg Time": "{:.3f}s",
        "Num Rules Avg": "{:.2f}",
        "Num Rules Std": "{:.2f}",
        "Num Preds Avg": "{:.2f}",
        "Num Preds Std": "{:.2f}"
    }
    
    # Explicitly list columns to apply bar charts to
    accuracy_cols = [col for col in final_summary_df.columns if col[1] == 'Avg Accuracy']
    brier_cols = [col for col in final_summary_df.columns if col[1] == 'Avg Brier Score']

    formatted_df = final_summary_df.style.format(format_dict) \
        .bar(subset=accuracy_cols, color='#5fba7d', vmin=0.5) \
        .bar(subset=brier_cols, color='#c9a1e8', vmin=0.5)
    
    print(f"\n--- EXPERIMENT SUMMARY: {fit_method_name} ---")
    display(formatted_df)
    
    # --- 5. Save the Summary to a CSV file ---
    final_summary_df.to_csv(filename)
    print(f"\nSummary results have been successfully saved to {filename}")


# --- Main Processing ---
# Create the initial "long format" DataFrame from the list of dictionaries
results_df_long = pd.DataFrame(results_list)

# Process FOLD-RM (fit) results
fold_rm_df = results_df_long[results_df_long['Fit Method'] == 'FOLD-RM (fit)'].copy()
process_and_display_results(fold_rm_df, "FOLD-RM (fit)", "fold_rm_metric_summary.csv")

# Process CON-FOLD (confidence_fit) results
con_fold_df = results_df_long[results_df_long['Fit Method'] == 'CON-FOLD (confidence_fit)'].copy()
process_and_display_results(con_fold_df, "CON-FOLD (confidence_fit)", "con_fold_metric_summary.csv")


--- EXPERIMENT SUMMARY: FOLD-RM (fit) ---



Summary results have been successfully saved to fold_rm_metric_summary.csv

--- EXPERIMENT SUMMARY: CON-FOLD (confidence_fit) ---



Summary results have been successfully saved to con_fold_metric_summary.csv
